In [1]:
import polars as pl

In [2]:
class CONFIG:
    target_col = "responder_6"
    lag_cols_original = ["date_id", "symbol_id"] + [
        f"responder_{idx}" for idx in range(9)
    ]
    lag_cols_rename = {
        f"responder_{idx}": f"responder_{idx}_lag_1" for idx in range(9)
    }
    valid_ratio = 0.05
    start_dt = 1100

In [3]:
train = (
    pl.scan_parquet(
        f"./input/jane-street-real-time-market-data-forecasting/train.parquet"
    )
    .select(
        pl.int_range(pl.len(), dtype=pl.UInt32).alias("id"),
        pl.all(),
    )
    .with_columns(
        (pl.col(CONFIG.target_col) * 2).cast(pl.Int32).alias("label"),
    )
    .filter(pl.col("date_id").gt(CONFIG.start_dt))
)

In [ ]:
lags = train.select(pl.col(CONFIG.lag_cols_original))
lags = lags.rename(CONFIG.lag_cols_rename)
lags = lags.with_columns(date_id=pl.col("date_id") + 1)
lags = lags.group_by(["date_id", "symbol_id"], maintain_order=True).last()
lags

In [5]:
train = train.join(lags, on=["date_id", "symbol_id"], how="left")
len_train = train.select(pl.col("date_id")).collect().shape[0]
valid_records = int(len_train * CONFIG.valid_ratio)
len_ofl_mdl = len_train - valid_records
last_tr_dt = train.select(pl.col("date_id")).collect().row(len_ofl_mdl)[0]
training_data = train.filter(pl.col("date_id").le(last_tr_dt))
validation_data = train.filter(pl.col("date_id").gt(last_tr_dt))

In [6]:
training_data.collect().write_parquet(
    f"./input/js24-preprocessing-create-lags/training.parquet",
    partition_by="date_id",
)

In [7]:
validation_data.collect().write_parquet(
    "./input/js24-preprocessing-create-lags/validation.parquet",
    partition_by="date_id",
)